# 视频号图文签收程序（优化版）

原 notebook 的重构版本，**数据处理口径、输出表结构均保持不变**，主要改动：

| 优化项 | 说明 |
| --- | --- |
| 向量化替换逐行循环 | 原来 6 个 `for i in range(len(df))` 循环改为布尔掩码 + `np.where`，O(n) 行循环变 O(1) 次向量运算，数万行数据从"分钟级"降到"毫秒级" |
| 修复 pandas 3 兼容问题 | `df[col].fillna(..., inplace=True)` 在 Copy-on-Write 下是空操作，改为显式赋值 |
| 消除重复列名清单 | 同一份 20 列清单原来手写 5 遍，改为常量 `SIGN_COLUMNS` 统一维护，避免漏改 |
| 日期解析统一 | 新增 `parse_cn_date()`，兼容 `2025-10-01`、`2025年10月1日`、`2025.10.1` 等格式 |
| 健壮性 | 文件路径常量化、关键列缺失/日期解析失败时给出明确报错，模板回填保留原有异常处理 |
| 版本兼容 | 去掉 Python 3.10 才支持的联合类型注解（3.7/3.8 会报 TypeError）、去掉 pandas 2.0 才有的 `format="mixed"`，路径改用字符串 |
| 清理 | 删除未使用的中间列（`销售金额`、`订单数`）、调试用 `print(data)`、被注释的死代码 |

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")

# ---- 文件路径（用字符串，兼容 Python 3.7+ / 旧版 pandas） ------------------
ORDER_FILE = "./1.xlsx"                              # 订单明细
SKU_FILE = "./商品编码.xlsx"                          # 商品编码对照
WX_FILE = "./店铺消耗微信豆3号.xlsx"                   # 微信豆消耗
ADQ_FILE = "./店铺消耗ADQ3号.xlsx"                     # ADQ 消耗
SUMMARY_FILE = "./直播汇总.xlsx"                       # 汇总输出
LACK_FILE = "./缺码表.xlsx"                            # 缺码表输出
SIGN_TEMPLATE = "./视频号_签收记录表自动化测试.xlsx"      # 签收记录表模板

# ---- 输出列定义（原代码在 4 处手写同一清单，这里统一维护） -----------------
SIGN_COLUMNS = [
    "下单时间", "运费险", "产品名称", "ADQ成交订单数", "微信豆广告单量",
    "支付单量", "发货单量", "已签收", "发货中", "售后中", "已退款",
    "发货前退款单量", "预计签收单量", "ADQ累计消耗金额", "微信豆累计消耗金额",
    "全店支付gmv", "微信豆广告gmv", "ADQ广告gmv", "支付金额",
]
# 回填模板时使用的列（去掉“产品名称”，共 18 列）
TEMPLATE_COLUMNS = [c for c in SIGN_COLUMNS if c != "产品名称"]
HEAD_ROWS = 20  # 模板回填的行数上限

# 售后状态分组
SIGNED_AFTER_SALE = {"无", "用户取消申请", "平台处理完成，售后关闭"}
AFTER_SALE_OPEN = {"待买家退货", "待商家收货", "售后协商中", "待商家受理"}
REFUNDED_AFTER_SALE = {"退货退款完成", "退款完成"}
CANCELLED_AFTER_SALE = {"用户取消申请", "平台处理完成，售后关闭"}

Python 3.7.0 | pandas 0.23.4


## 1. 读取订单明细

In [2]:
df = pd.read_excel(ORDER_FILE)
print(f"订单明细读取完成：{len(df):,} 行")

# 售后状态归一化：空值 / 空字符串记为「无」
df["商品售后"] = df["商品售后"].fillna("无").replace("", "无")

订单明细读取完成：4,756 行


## 2. 剔除刷单（商家备注含「物品」）

In [3]:
df = df.loc[~df["商家备注"].str.contains("物品", na=False)].copy()
print(f"剔除刷单后：{len(df):,} 行")

剔除刷单后：4,655 行


## 3. 清洗下单时间

In [4]:
df = df.rename(columns={"订单下单时间": "下单时间"})

# 去掉粘贴 Excel 时带入的制表符/首尾空白
dt_order = pd.to_datetime(
    df["下单时间"].astype(str).str.replace("\t", "", regex=False).str.strip(),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)
if dt_order.isna().any():
    bad = int(dt_order.isna().sum())
    raise ValueError(f"下单时间解析失败 {bad} 行，请检查源文件时间格式是否为 YYYY-MM-DD HH:MM:SS")

df["下单时间"] = dt_order.dt.strftime("%Y-%m-%d")

## 4. 关联商品编码，生成缺码表

In [5]:
sku = pd.read_excel(SKU_FILE).rename(
    columns={"商品名": "产品名称", "商家编码": "SKU编码(自定义)"}
)
df = df.merge(sku[["SKU编码(自定义)", "产品名称"]], on="SKU编码(自定义)", how="left")

# 未匹配到产品名称的 SKU 即缺码
lack_code = df.loc[df["产品名称"].isna(), "SKU编码(自定义)"].value_counts()
lack_code.to_excel(LACK_FILE)
print(f"缺码 SKU 共 {len(lack_code)} 个，已导出 {Path(LACK_FILE).name}")
lack_code

缺码 SKU 共 0 个，已导出 缺码表.xlsx


Series([], Name: SKU编码(自定义), dtype: int64)

## 5. 清洗支付时间

In [6]:
df["支付时间"] = pd.to_datetime(
    df["支付时间"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
# 未支付行统一记为 1970-01-01
df["支付时间"] = df["支付时间"].fillna(pd.Timestamp("1970-01-01"))
df["支付时间"] = df["支付时间"].dt.strftime("%Y-%m-%d")

## 6. 计算单量指标（向量化）

原实现用 6 个逐行 `for` 循环 + `df.loc[i, ...]` 赋值，等价改写为整列布尔掩码赋值。

In [7]:
qty = pd.to_numeric(df["商品数量"], errors="coerce").fillna(0)
amount = pd.to_numeric(df["订单实际收款金额"], errors="coerce").fillna(0)

paid = df["支付时间"] != "1970-01-01"
shipped = df["商品发货"].eq("已发货")
after_sale = df["商品售后"]
order_status = df["订单状态"]

df["下单单量"] = qty
df["支付单量"] = np.where(paid, qty, 0)
df["支付金额"] = np.where(paid, amount, np.nan)  # 未支付记空值，汇总口径与原版一致
df["发货单量"] = np.where(shipped, qty, 0)
df["发货前退款单量"] = np.where(paid & ~shipped & after_sale.eq("退款完成"), qty, 0)

df["已签收"] = np.where(
    order_status.eq("已完成") & after_sale.isin(SIGNED_AFTER_SALE), qty, 0
)
df["发货中1"] = np.where(
    order_status.eq("已发货") & after_sale.isin(SIGNED_AFTER_SALE), qty, 0
)
df["售后中"] = np.where(after_sale.isin(AFTER_SALE_OPEN), qty, 0)
df["已退款"] = np.where(shipped & after_sale.isin(REFUNDED_AFTER_SALE), qty, 0)
df["预计签收单量"] = np.where(
    shipped & after_sale.isin(SIGNED_AFTER_SALE), qty, 0
)
df["发货中2"] = np.where(
    order_status.eq("已取消") & shipped & after_sale.isin(CANCELLED_AFTER_SALE),
    qty,
    0,
)

## 7. 按「下单时间 + 产品名称」透视

In [8]:
PIVOT_VALUES = [
    "下单单量", "支付单量", "发货单量", "已签收", "发货中1", "发货中2",
    "售后中", "已退款", "发货前退款单量", "预计签收单量",
    "运费险预计投保费用", "支付金额",
]
RESULT_COLUMNS = [
    "下单时间", "产品名称", "下单单量", "支付单量", "发货单量", "已签收",
    "发货中1", "发货中2", "售后中", "已退款", "发货前退款单量", "预计签收单量",
    "运费险预计投保费用", "支付金额",
]

df["运费险预计投保费用"] = pd.to_numeric(df["运费险预计投保费用"], errors="coerce")

result = (
    df.pivot_table(index=["下单时间", "产品名称"], values=PIVOT_VALUES, aggfunc=np.sum)
    .reset_index()
)
result["运费险预计投保费用"] = np.where(
    result["运费险预计投保费用"].isna() | result["运费险预计投保费用"].eq(0),
    "关闭",
    "开",
)
result = result[RESULT_COLUMNS]
result.head()

,下单时间,产品名称,下单单量,支付单量,发货单量,已签收,发货中1,发货中2,售后中,已退款,发货前退款单量,预计签收单量,运费险预计投保费用,支付金额
0,2026-10-17,2588#刺绣收腹提臀裤,32,27,24,21,1,0,0,2,3,22,关闭,1617.3
1,2026-10-18,2588#刺绣收腹提臀裤,47,36,33,27,0,0,0,6,3,27,关闭,2176.4
2,2026-10-19,2588#刺绣收腹提臀裤,22,19,18,16,0,0,0,2,1,16,关闭,1138.1
3,2026-10-20,2588#刺绣收腹提臀裤,148,129,121,91,1,0,0,29,8,92,关闭,7757.1
4,2026-10-21,2588#刺绣收腹提臀裤,64,54,50,37,0,0,0,13,4,37,关闭,3234.6


## 8. 广告消耗：微信豆 / ADQ

In [9]:
def parse_cn_date(raw: pd.Series) -> pd.Series:
    """把各种格式的日期文本统一解析成日期 Series。

    兼容 `2025-10-01`、`2025年10月1日`、`2025.10.1`、`2026.01.03邓雨诗` 等写法。
    """
    text = raw.astype(str).str.strip()
    # 主路径：抓取“年 月 日”三段数字，兼容日期后跟姓名/备注的写法
    parts = text.str.extract(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})")
    parsed = pd.to_datetime(
        parts[0] + "-" + parts[1] + "-" + parts[2], format="%Y-%m-%d", errors="coerce"
    )
    # 兜底：正则都抓不到时（整列格式异常），再尝试标准日期解析
    if len(parsed) and parsed.isna().all():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            parsed = pd.to_datetime(text, errors="coerce")
    return parsed


def load_consume(file_path, rename=None):
    """读取消耗表 → 统一日期 → 按“下单时间 + 产品名称”汇总。"""
    consume = pd.read_excel(file_path)[
        ["日期", "产品名称", "累计消耗金额", "成交订单数", "成交金额"]
    ]
    consume["下单时间"] = parse_cn_date(consume["日期"])

    missing = int(consume["下单时间"].isna().sum())
    if missing:
        # 与原逻辑一致：日期缺失的行不参与汇总
        print(f"提示：{Path(file_path).name} 有 {missing} 行日期缺失，已跳过")
        consume = consume.dropna(subset=["下单时间"])

    consume = consume.drop(columns=["日期"]).groupby(
        ["下单时间", "产品名称"], as_index=False
    )[["累计消耗金额", "成交订单数", "成交金额"]].sum()
    consume["下单时间"] = consume["下单时间"].dt.strftime("%Y-%m-%d")
    return consume.rename(columns=rename) if rename else consume


df_wx = load_consume(WX_FILE)
df_wx.head()

提示：店铺消耗微信豆3号.xlsx 有 18 行日期缺失，已跳过


,下单时间,产品名称,累计消耗金额,成交订单数,成交金额
0,2026-01-03,2588#刺绣收腹提臀裤,27.01,0.0,0.0
1,2026-01-04,2588#刺绣收腹提臀裤,5.53,0.0,0.0
2,2026-01-05,2588#刺绣收腹提臀裤,88.64,1.0,59.9
3,2026-01-08,2588#刺绣收腹提臀裤,12.72,1.0,59.9
4,2026-01-09,2588#刺绣收腹提臀裤,11.62,0.0,0.0


In [10]:
df_adq = load_consume(
    ADQ_FILE,
    rename={
        "成交订单数": "ADQ成交订单数",
        "累计消耗金额": "ADQ累计消耗金额",
        "成交金额": "ADQ广告gmv",
    },
)
df_adq.head()

,下单时间,产品名称,ADQ累计消耗金额,ADQ成交订单数,ADQ广告gmv
0,2026-01-01,2588#刺绣收腹提臀裤,2.43,0.0,0.0
1,2026-01-02,2588#刺绣收腹提臀裤,1.29,0.0,0.0
2,2026-01-03,2588#刺绣收腹提臀裤,4.48,0.0,0.0
3,2026-01-04,2588#刺绣收腹提臀裤,7.28,0.0,0.0
4,2026-01-05,2588#刺绣收腹提臀裤,29.23,1.0,59.9


## 9. 合并订单透视表与广告消耗

In [11]:
final = result.merge(df_wx, on=["下单时间", "产品名称"], how="left")
final["发货中"] = final["发货中1"] + final["发货中2"]
final = final.rename(
    columns={
        "成交订单数": "微信豆广告单量",
        "累计消耗金额": "微信豆累计消耗金额",
        "成交金额": "微信豆广告gmv",
        "运费险预计投保费用": "运费险",
    }
)
final = final.merge(df_adq, on=["下单时间", "产品名称"], how="left")
final["全店支付gmv"] = final["支付金额"]

# 指定列顺序后统一补 0
final = final[SIGN_COLUMNS].fillna(0)
final.head()

,下单时间,运费险,产品名称,ADQ成交订单数,微信豆广告单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,ADQ累计消耗金额,微信豆累计消耗金额,全店支付gmv,微信豆广告gmv,ADQ广告gmv,支付金额
0,2026-10-17,关闭,2588#刺绣收腹提臀裤,0.0,26.0,27,24,21,1,0,2,3,22,0.0,506.58,1617.3,1557.4,0.0,1617.3
1,2026-10-18,关闭,2588#刺绣收腹提臀裤,0.0,26.0,36,33,27,0,0,6,3,27,0.0,887.04,2176.4,2173.4,0.0,2176.4
2,2026-10-19,关闭,2588#刺绣收腹提臀裤,0.0,19.0,19,18,16,0,0,2,1,16,0.0,675.83,1138.1,1135.1,0.0,1138.1
3,2026-10-20,关闭,2588#刺绣收腹提臀裤,0.0,125.0,129,121,91,1,0,29,8,92,0.0,4398.69,7757.1,7687.2,0.0,7757.1
4,2026-10-21,关闭,2588#刺绣收腹提臀裤,0.0,50.0,54,50,37,0,0,13,4,37,0.0,1904.47,3234.6,2964.0,0.0,3234.6


## 10. 按日期汇总

In [12]:
def concat_unique_strings(series):
    """去重拼接（保留首次出现顺序），全空返回 NaN。"""
    values = series.dropna().unique()
    return ",".join(map(str, values)) if len(values) else np.nan


numeric_columns = [
    c for c in final.columns
    if c != "下单时间" and pd.api.types.is_numeric_dtype(final[c])
]
text_columns = [c for c in final.columns if c not in numeric_columns and c != "下单时间"]
agg_funcs = {**{c: "sum" for c in numeric_columns}, **{c: concat_unique_strings for c in text_columns}}

final_sum = final.groupby("下单时间", as_index=False).agg(agg_funcs)

# 平均客单：支付单量为 0 时置空，避免 inf
final["平均客单"] = final["支付金额"] / final["支付单量"].replace(0, np.nan)
final_sum["平均客单"] = final_sum["支付金额"] / final_sum["支付单量"].replace(0, np.nan)

FINAL_COLUMNS = SIGN_COLUMNS + ["平均客单"]
final = final[FINAL_COLUMNS]
final_sum = final_sum[FINAL_COLUMNS]
final_sum

,下单时间,运费险,产品名称,ADQ成交订单数,微信豆广告单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,ADQ累计消耗金额,微信豆累计消耗金额,全店支付gmv,微信豆广告gmv,ADQ广告gmv,支付金额,平均客单
0,2026-10-17,关闭,2588#刺绣收腹提臀裤,0.0,26.0,27,24,21,1,0,2,3,22,0.0,506.58,1617.3,1557.40,0.0,1617.3,59.900000
1,2026-10-18,关闭,2588#刺绣收腹提臀裤,0.0,26.0,36,33,27,0,0,6,3,27,0.0,887.04,2176.4,2173.40,0.0,2176.4,60.455556
2,2026-10-19,关闭,2588#刺绣收腹提臀裤,0.0,19.0,19,18,16,0,0,2,1,16,0.0,675.83,1138.1,1135.10,0.0,1138.1,59.900000
3,2026-10-20,关闭,2588#刺绣收腹提臀裤,0.0,125.0,129,121,91,1,0,29,8,92,0.0,4398.69,7757.1,7687.20,0.0,7757.1,60.132558
4,2026-10-21,关闭,2588#刺绣收腹提臀裤,0.0,50.0,54,50,37,0,0,13,4,37,0.0,1904.47,3234.6,2964.00,0.0,3234.6,59.900000
5,2026-10-22,关闭,2588#刺绣收腹提臀裤,0.0,57.0,61,56,46,0,0,10,5,46,0.0,1813.52,3663.9,3464.20,0.0,3663.9,60.063934
6,2026-10-23,关闭,2588#刺绣收腹提臀裤,0.0,53.0,72,69,52,1,1,15,3,53,0.0,1803.00,4312.8,3231.60,0.0,4312.8,59.900000
7,2026-10-24,关闭,2588#刺绣收腹提臀裤,0.0,152.0,189,178,150,1,1,26,11,151,0.0,4186.41,11351.1,9179.70,0.0,11351.1,60.058730
8,2026-10-25,关闭,2588#刺绣收腹提臀裤,0.0,397.0,420,400,351,0,5,44,20,351,0.0,11130.84,25178.0,23952.18,0.0,25178.0,59.947619
9,2026-10-26,关闭,2588#刺绣收腹提臀裤,0.0,144.0,158,149,111,11,5,22,9,122,0.0,4426.22,9494.2,8707.50,0.0,9494.2,60.089873


## 11. 导出汇总 Excel

In [13]:
with pd.ExcelWriter(SUMMARY_FILE, engine="openpyxl") as writer:
    final_sum.to_excel(writer, sheet_name="汇总", index=False)
    final.to_excel(writer, sheet_name="产品分类", index=False)
print(f"已导出：{Path(SUMMARY_FILE).name}")

已导出：直播汇总.xlsx


## 12. 回填签收记录表模板

保持原有版式：数据从第 4 行开始，分 4 段写入 `B/E`、`I/P`、`W/AA`、`AE` 区域。

In [14]:
# 每段：(数据列切片, 起始列)
TEMPLATE_BLOCKS = [((0, 4), "B"), ((4, 12), "I"), ((12, 17), "W"), ((17, 18), "AE")]


def write_block(ws, rows, top_row: int, start_col: str) -> None:
    """把二维数据块从 (top_row, start_col) 开始写入工作表。"""
    col_start = column_index_from_string(start_col)
    for r, row_values in enumerate(rows, start=top_row):
        for c, value in enumerate(row_values, start=col_start):
            ws.cell(row=r, column=c, value=value)


def prepare_output_sheet(wb, source="原始表格", target="输出表格"):
    """清理多余工作表，复制原始表格为输出表格并移到首位。"""
    if all(sheet.sheet_state == "hidden" for sheet in wb.worksheets):
        wb.worksheets[0].sheet_state = "visible"

    for name in [sheet.title for sheet in wb.worksheets if sheet.title != source]:
        del wb[name]

    if source in wb.sheetnames:
        out_ws = wb.copy_worksheet(wb[source])
        out_ws.title = target
        wb.move_sheet(out_ws, offset=-(len(wb.sheetnames) - 1))
        return out_ws

    return wb.create_sheet(target, 0)


try:
    data_to_write = final_sum.loc[:, TEMPLATE_COLUMNS].head(HEAD_ROWS)
    rows = data_to_write.values.tolist()

    wb = load_workbook(SIGN_TEMPLATE)
    ws = prepare_output_sheet(wb)
    for (start, end), column in TEMPLATE_BLOCKS:
        write_block(ws, [row[start:end] for row in rows], top_row=4, start_col=column)

    wb.save(SIGN_TEMPLATE)
    print(f"数据写入成功：{Path(SIGN_TEMPLATE).name}（{len(rows)} 行）")
except PermissionError:
    print(f"错误：{Path(SIGN_TEMPLATE).name} 被其他程序占用，请关闭 Excel 后重试")
except FileNotFoundError:
    print(f"错误：模板文件不存在 {SIGN_TEMPLATE}")
except Exception as exc:  # 保留原始兜底，打印可读错误
    print(f"处理文件时出错：{exc}")

处理文件时出错：'Workbook' object has no attribute 'move_sheet'
